In [ ]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import Bio
import Bio.SeqIO

import utils

In [ ]:
datadir = "data/jared_PacBio_data_2"
bindir = "/mnt/scratch/sameer/ordinal"
binfile = f"{bindir}/bin5_stats.tsv"

In [ ]:
WT_fasta = Bio.SeqIO.read(datadir + "/2D.fasta", format="fasta")
WT_DNA = WT_fasta.seq
WT = WT_DNA.translate()

In [ ]:
df = pd.read_csv(binfile, sep="\t")
df["tilde_count"] = df.qual.str.count("~") # number of high quality bases
df.head()

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(13, 4), sharey=True )
sns.histplot(data=df, x="np", ax=axs[0])
axs[0].set_xlabel("Number of read passes for seq")
axs[1].hist(df.length, bins=50)
axs[1].set_xlabel("Length of read seq")
sns.histplot(data=df, x="tilde_count", ax=axs[2])
axs[2].set_xlabel("Number of high quality bases in seq")
plt.suptitle("CCS stats")
pass

In [ ]:
quality_seqs_mask = (df.np == 60) \
    & (df.tilde_count >= df.tilde_count.mode().item()) \
    & (df.length >= len(WT_DNA) \
    & (df.length <= len(WT_DNA)* 1.5)  )

# quality_seqs_mask = (df.np >= 0) \
#     & (df.tilde_count >= 0) \
#     & (df.length >= len(WT_DNA) \
#     & (df.length <= len(WT_DNA)* 1.5)  )

df_quality = df[quality_seqs_mask]
len(df_quality)

In [ ]:
df_quality_agg = df_quality.groupby("seq").agg(
        cnt=pd.NamedAgg(column="seq", aggfunc=len),
        ref_dist=pd.NamedAgg(column="ref_dist", 
                    aggfunc=lambda x: x.head(1).item())
    ).reset_index().sort_values("cnt", ascending=False).reset_index(drop=True)
df_quality_agg

In [ ]:
plt.plot(df_quality_agg.index + 1, df_quality_agg.cnt)
plt.xscale('log')
plt.yscale('log')
plt.ylabel("Read Count")
plt.xlabel("Sequence index")
plt.title("Log-Log plot of reads vs seqence index")
WT_index = df_quality_agg.index[df_quality_agg.ref_dist == 0].item()
ax = plt.gca()
ax.plot(WT_index + 1, df_quality_agg.iloc[WT_index].cnt, "xr", label="wild-type")
plt.legend()
pass

In [ ]:
# Amino acid distance from wild-type
df_quality_agg["ref_aa_dist"] = df_quality_agg.seq.map(
        lambda x: utils.hamming_dist(Bio.Seq.Seq(x).translate(), WT))

In [ ]:
df_quality_agg[:16]

In [ ]:
# with open("/tmp/ordinal_bin5.fasta", "wt") as fh_out:
#     for i, seq in enumerate(df_quality_agg.seq[:16]):
#         print(f">s{i+1}\n{seq}", file=fh_out)